In [23]:
from __future__ import annotations

In [24]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, RobustScaler

In [25]:
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from torch.cuda.amp import GradScaler, autocast

In [26]:
from IPython.display import display
from tqdm.auto import tqdm
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [29]:
from catboost import CatBoostRegressor, Pool

In [30]:
df = pd.read_csv("data_for_models/df_4_full.zip")

In [31]:
df_full = df.copy()

In [32]:
df_full = df_full.rename(columns={"address.city_new": "address_city_new"})

In [34]:
# --- Таргет и группировка CV ---
TARGET = "salary_from_log"
GROUP_COL = "region_name"

# --- Числовые блоки 
COLS_TRIG = ["lat_sin", "lat_cos", "lon_sin", "lon_cos"]
COLS_BINARY = ["geo_available", "distance_missing", "macro_k_u_imputed"]
COLS_DISTANCE = ["distance_to_reg_center"]
COLS_MACRO = ["RK", "GRP_K", "U_delta"]

# Без scaler: тригонометрия + бинарные индикаторы
COLS_BLOCK_A = COLS_TRIG + COLS_BINARY

#  Категориальные: OHE (низкая/средняя кардинальность)
COLS_CAT_OHE = [
    "role_name",
    "schedule_id",
    "employment_id",
    "economic_region",
    "region_name",
    "experience_ord",
]

# -Высокая кардинальность, индексы для nn.Embedding (словарь fit на train) 
COLS_CAT_EMBED = ["address_city_new", "geohash_4", "geohash_5", "geohash_6"]

COLS_TEXT = ["text_for_model"]
COLS_SERVICE = ["row_id", "salary_from_adj"]

# Все колонки, которые участвуют в табличном sklearn-пайплайне (без текста и сервиса)
COLS_TABULAR_SKLEARN = COLS_BLOCK_A + COLS_DISTANCE + COLS_MACRO + COLS_CAT_OHE

REQUIRED_BASE = (
    COLS_TABULAR_SKLEARN + COLS_CAT_EMBED + COLS_TEXT + [TARGET, GROUP_COL] + COLS_SERVICE
)
REQUIRED_BASE = list(dict.fromkeys(REQUIRED_BASE))

In [35]:
# M2.10.2_geo_macro: витрина для CatBoost (сырые колонки, без sklearn OHE)
COLS_CB_FLOAT = COLS_TRIG + COLS_BINARY + COLS_DISTANCE + COLS_MACRO
COLS_CB_CAT = [
    "role_name", "schedule_id", "employment_id", "economic_region", "region_name",
    "experience_ord", "address_city_new", "geohash_4", "geohash_5", "geohash_6",
]
COLS_CB_TABULAR = COLS_CB_FLOAT + COLS_CB_CAT

In [36]:
SENTINEL_DISTANCE = -999.0


def apply_fixed_vitrine_rules(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Триггеры: нет гео → нули по всем четырём (параллельно geo_available)
    miss_geo = out["geo_available"].eq(0) | out[COLS_TRIG].isna().any(axis=1)
    out.loc[miss_geo, COLS_TRIG] = 0.0

    # Расстояние: если всё ещё NaN — сентинел + флаг 
    d_na = out["distance_to_reg_center"].isna()
    if d_na.any():
        out.loc[d_na, "distance_to_reg_center"] = SENTINEL_DISTANCE
        out.loc[d_na, "distance_missing"] = 1.0

    # Типы для блока A
    for c in COLS_BINARY:
        out[c] = out[c].astype("float32")
    out[COLS_TRIG] = out[COLS_TRIG].astype("float32")

    return out


def qa_vitrine_before_split(df: pd.DataFrame) -> None:
    """Проверки до любого fit scaler/OHE."""
    for c in REQUIRED_BASE:
        if c not in df.columns:
            raise KeyError(f"Нет колонки {c}")

    assert df[TARGET].notna().all(), "Пропуски в таргете"
    assert df[GROUP_COL].notna().all(), "Пропуски в region_name"

    assert df[COLS_TRIG].notna().all().all(), "После правил не должно остаться NaN в sin/cos"
    assert df["distance_to_reg_center"].notna().all(), "После правил не должно остаться NaN в расстоянии"

    for c in COLS_BINARY:
        u = set(np.unique(df[c].to_numpy()))
        assert u.issubset({0.0, 1.0}), f"{c} должен быть бинарным 0/1, уникальные: {u}"

    assert df["text_for_model"].astype(str).str.strip().str.len().gt(0).all()

    # OHE-категории: пропуски лучше явно обработать до OHE
    for c in COLS_CAT_OHE:
        if df[c].isna().any():
            raise ValueError(f"В колонке {c} есть NaN — задайте политику (например, строка 'missing').")


df_full = apply_fixed_vitrine_rules(df_full)
qa_vitrine_before_split(df_full)

In [37]:
def _macro_log1p_then_stack(X: np.ndarray) -> np.ndarray:
    """X: (n, 3) порядок столбцов RK, GRP_K, U_delta."""
    X = np.asarray(X, dtype=np.float64)
    out = np.empty_like(X)
    out[:, 0] = X[:, 0]
    out[:, 1] = np.log1p(np.clip(X[:, 1], 0.0, None))
    out[:, 2] = X[:, 2]
    return out


def build_tabular_sklearn_preprocessor(
    ohe_categories: list[np.ndarray] | None = None,
) -> ColumnTransformer:
    macro_pipe = Pipeline(
        steps=[
            ("log1p_grp", FunctionTransformer(_macro_log1p_then_stack, validate=False)),
            ("robust", RobustScaler()),
        ]
    )

    ohe_kw: dict = dict(
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32,
    )
    if ohe_categories is not None:
        ohe_kw["categories"] = ohe_categories

    ohe = OneHotEncoder(**ohe_kw)

    return ColumnTransformer(
        transformers=[
            ("block_a", "passthrough", COLS_BLOCK_A),
            ("block_b", RobustScaler(), COLS_DISTANCE),
            ("block_c", macro_pipe, COLS_MACRO),
            ("cat_ohe", ohe, COLS_CAT_OHE),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


In [38]:
@dataclass
class EmbeddingVocab:
    """token -> id, 0 зарезервирован для UNK/OOV."""

    col: str
    token_to_id: dict[str, int]

    @classmethod
    def fit(cls, series: pd.Series, col: str, min_count: int = 1) -> "EmbeddingVocab":
        vc = series.astype(str).fillna("__NA__").value_counts()
        tokens = vc[vc >= min_count].index.tolist()
        token_to_id = {"__UNK__": 0}
        for i, t in enumerate(tokens, start=1):
            token_to_id[str(t)] = i
        return cls(col=col, token_to_id=token_to_id)

    def transform(self, series: pd.Series) -> np.ndarray:
        s = series.astype(str).fillna("__NA__")
        unk = self.token_to_id["__UNK__"]
        return s.map(lambda x: self.token_to_id.get(str(x), unk)).to_numpy(np.int64)


def fit_embedding_vocabs(
    df_train: pd.DataFrame,
    cols: Iterable[str] = COLS_CAT_EMBED,
    min_count: int = 1,
) -> dict[str, EmbeddingVocab]:
    return {c: EmbeddingVocab.fit(df_train[c], c, min_count=min_count) for c in cols}

In [39]:
def build_ohe_categories_from_pool(
    df_pool: pd.DataFrame,
    cols: list[str],
) -> list[np.ndarray]:
    """
    Уровни категорий по всему train-pool (без df_test).
    Размерность OHE после fit на любом train-фолде будет совпадать с transform на val.
    """
    categories: list[np.ndarray] = []
    for c in cols:
        s = df_pool[c]
        if pd.api.types.is_bool_dtype(s):
            s = s.astype(int)
        if pd.api.types.is_integer_dtype(s) or pd.api.types.is_float_dtype(s):
            vals = np.sort(np.unique(s.to_numpy()))
            vals = vals[~np.isnan(vals)] if np.issubdtype(vals.dtype, np.floating) else vals
            cats = vals.tolist()
            if len(cats) == 0:
                cats = [0]
            categories.append(np.asarray(cats, dtype=np.int64))
        else:
            cats = sorted(s.astype(str).fillna("__NA__").unique().tolist())
            if len(cats) == 0:
                cats = ["__NA__"]
            categories.append(np.asarray(cats, dtype=object))
    return categories

In [40]:
def train_test_indices_within_region(
    df: pd.DataFrame,
    *,
    region_col: str = GROUP_COL,
    test_frac: float = 0.2,
    random_state: int = 42,
) -> tuple[np.ndarray, np.ndarray]:
    """Возвращает индексы train и test (строки df). В каждом регионе ~test_frac в test, минимум 1 в train при n>=2."""
    rng = np.random.default_rng(random_state)
    train_ix: list[int] = []
    test_ix: list[int] = []

    for _, grp in df.groupby(region_col, sort=False):
        idx = grp.index.to_numpy()
        rng.shuffle(idx)
        n = len(idx)
        if n == 0:
            continue
        if n == 1:
            train_ix.append(int(idx[0]))
            continue
        n_test = int(np.round(n * test_frac))
        n_test = max(1, min(n - 1, n_test))
        test_ix.extend(idx[:n_test].tolist())
        train_ix.extend(idx[n_test:].tolist())

    return np.array(train_ix, dtype=np.int64), np.array(test_ix, dtype=np.int64)


train_idx, test_idx = train_test_indices_within_region(
    df_full, test_frac=0.2, random_state=42
)
df_train_pool = df_full.loc[train_idx].reset_index(drop=True)
df_test = df_full.loc[test_idx].reset_index(drop=True)

# Фиксированные уровни OHE и словари embedding — только по train-pool (test не смотрим)
OHE_CATEGORIES = build_ohe_categories_from_pool(df_train_pool, COLS_CAT_OHE)
EMB_VOCABS_POOL = fit_embedding_vocabs(df_train_pool, COLS_CAT_EMBED, min_count=1)

# GroupKFold только по train_pool (test уже отложен)
gkf = GroupKFold(n_splits=5)
X_pool = df_train_pool[COLS_TABULAR_SKLEARN]
y_pool = df_train_pool[TARGET].to_numpy()
groups = df_train_pool[GROUP_COL].to_numpy()

n_ohe_features = None

for fold, (tr, va) in enumerate(gkf.split(X_pool, y_pool, groups)):
    X_tr, X_va = X_pool.iloc[tr], X_pool.iloc[va]
    pre = build_tabular_sklearn_preprocessor(ohe_categories=OHE_CATEGORIES)
    pre.fit(X_tr)
    Z_tr = pre.transform(X_tr)
    Z_va = pre.transform(X_va)
    assert Z_tr.shape[1] == Z_va.shape[1], (fold, Z_tr.shape, Z_va.shape)
    if n_ohe_features is None:
        n_ohe_features = Z_tr.shape[1]

    print(fold, Z_tr.shape, Z_va.shape)
print("Фиксированная размерность табличного вектора:", n_ohe_features)

0 (84215, 133) (21052, 133)
1 (84212, 133) (21055, 133)
2 (84212, 133) (21055, 133)
3 (84216, 133) (21051, 133)
4 (84213, 133) (21054, 133)
Фиксированная размерность табличного вектора: 133


In [41]:
from catboost import CatBoostRegressor, Pool
def default_m210_catboost_params() -> dict:
    """Гиперпараметры CatBoost в духе M2.10 (pipeline_project.md)."""
    params: dict = {
        "iterations": 1000,
        "learning_rate": 0.05,
        "depth": 6,
        "loss_function": "RMSE",
        "random_seed": 42,
        "verbose": False,
        "allow_writing_files": False,
    }
    if torch.cuda.is_available():
        params["task_type"] = "GPU"
    return params
def _cb_pool_X(df: pd.DataFrame) -> tuple[pd.DataFrame, list[int]]:
    """Матрица признаков для CatBoost: сначала числовые, затем категориальные; индексы cat_features — хвост."""
    for c in COLS_CB_TABULAR:
        if c not in df.columns:
            raise KeyError(f"Нет колонки {c!r} для CatBoost (M2.10 витрина)")
    X = df[COLS_CB_TABULAR].copy()
    cat_idx = list(range(len(COLS_CB_FLOAT), len(COLS_CB_FLOAT) + len(COLS_CB_CAT)))
    return X, cat_idx
def catboost_m210_pred_for_train_val(
    df_tr: pd.DataFrame,
    df_va: pd.DataFrame,
    *,
    y_col: str = TARGET,
    group_col: str = GROUP_COL,
    inner_splits: int = 4,
    catboost_params: dict | None = None,
    oof_on_train: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    """Табличная ветка M2.10: один внешний train / val.
    Возвращает (pred_train, pred_val), оба float32, длины len(df_tr) и len(df_va).
    - pred_val: CatBoost, обученный на полном df_tr, предсказание для df_va.
    - pred_train: если oof_on_train — OOF по GroupKFold(group_col) внутри df_tr;
      иначе in-sample predict на df_tr (без внутреннего CV).
    """
    params = dict(catboost_params or default_m210_catboost_params())
    y_tr = df_tr[y_col].to_numpy(dtype=np.float32)
    X_tr, cat_idx = _cb_pool_X(df_tr)
    X_va, _ = _cb_pool_X(df_va)
    if oof_on_train and len(df_tr) >= 10:
        groups = df_tr[group_col].to_numpy()
        n_groups = int(np.unique(groups).size)
        n_splits = max(2, min(int(inner_splits), n_groups))
        gkf_inner = GroupKFold(n_splits=n_splits)
        pred_train = np.zeros(len(df_tr), dtype=np.float32)
        for tr_in, va_in in gkf_inner.split(np.zeros(len(df_tr)), y_tr, groups):
            pool_in = Pool(
                X_tr.iloc[tr_in],
                label=y_tr[tr_in],
                cat_features=cat_idx,
            )
            m_inner = CatBoostRegressor(**params)
            m_inner.fit(pool_in)
            pred_train[va_in] = m_inner.predict(X_tr.iloc[va_in]).astype(np.float32)
    else:
        pool_tr = Pool(X_tr, label=y_tr, cat_features=cat_idx)
        m_tr = CatBoostRegressor(**params)
        m_tr.fit(pool_tr)
        pred_train = m_tr.predict(X_tr).astype(np.float32)
    pool_full = Pool(X_tr, label=y_tr, cat_features=cat_idx)
    m_val = CatBoostRegressor(**params)
    m_val.fit(pool_full)
    pred_val = m_val.predict(X_va).astype(np.float32)
    return pred_train, pred_val

In [42]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BERT_NAME = "DeepPavlov/rubert-base-cased"
MAX_LENGTH = 512

In [43]:
class VacancyFoldDataset(Dataset):
    """Один фолд: pred_cb — скаляр CatBoost (M2.10) по таблице; текст и emb — как раньше."""
    def __init__(
        self,
        pred_cb: np.ndarray,
        texts: list[str],
        emb_idx: dict[str, np.ndarray],
        y: np.ndarray,
        tokenizer,
        max_length: int,
    ):
        assert len(pred_cb) == len(texts) == len(y)
        self.pred_cb = np.asarray(pred_cb, dtype=np.float32).reshape(-1)
        self.texts = texts
        self.emb_idx = emb_idx
        self.y = y.astype(np.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length
        for c, arr in emb_idx.items():
            assert len(arr) == len(texts), c
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i: int):
        enc = self.tokenizer(
            self.texts[i],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        item = {
            "pred_cb": torch.tensor([self.pred_cb[i]], dtype=torch.float32),
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "y": torch.tensor(self.y[i], dtype=torch.float32),
        }
        for c in self.emb_idx:
            item[f"emb_{c}"] = torch.tensor(self.emb_idx[c][i], dtype=torch.long)
        return item
def collate_fn(batch: list[dict]) -> dict:
    keys = [k for k in batch[0] if k != "y"]
    out = {k: torch.stack([b[k] for b in batch], dim=0) for k in keys}
    out["y"] = torch.stack([b["y"] for b in batch], dim=0)
    return out

In [44]:
class MultimodalSalaryModel(nn.Module):
    """ruBERT: смесь последних K слоёв + self-attention; pooling mean | cls;
    слияние [z_txt | pred_cb | z_emb] → MLP (таблица только через скаляр CatBoost)."""
    def __init__(
        self,
        bert_name: str,
        vocab_sizes: dict[str, int],
        emb_dim: int = 16,
        hidden: int = 256,
        dropout: float = 0.2,
        freeze_bert: bool = False,
        *,
        text_last_k_layers: int = 4,
        text_attn_heads: int = 8,
        text_attn_dropout: float = 0.1,
        text_pooling: str = "mean",
    ):
        super().__init__()
        if text_pooling not in ("mean", "cls"):
            raise ValueError("text_pooling must be 'mean' or 'cls'")
        self.text_pooling = text_pooling
        self.text_last_k_layers = int(text_last_k_layers)
        self.bert = AutoModel.from_pretrained(bert_name)
        for p in self.bert.parameters():
            p.requires_grad = not freeze_bert
        d_bert = self.bert.config.hidden_size
        n_enc = self.bert.config.num_hidden_layers
        if self.text_last_k_layers < 1 or self.text_last_k_layers > n_enc:
            raise ValueError(f"text_last_k_layers must be in [1, {n_enc}]")
        self.layer_mix_logits = nn.Parameter(torch.zeros(self.text_last_k_layers))
        self.text_attn = nn.MultiheadAttention(
            embed_dim=d_bert,
            num_heads=int(text_attn_heads),
            dropout=float(text_attn_dropout),
            batch_first=True,
        )
        self.text_attn_norm = nn.LayerNorm(d_bert)
        self.text_attn_dropout = nn.Dropout(float(text_attn_dropout))
        self.embeds = nn.ModuleDict(
            {k: nn.Embedding(n, emb_dim, padding_idx=0) for k, n in vocab_sizes.items()}
        )
        d_emb = emb_dim * len(vocab_sizes)
        d_cb = 1
        fuse_in = d_bert + d_cb + d_emb
        self.head = nn.Sequential(
            nn.Linear(fuse_in, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )
    def forward(self, batch: dict) -> torch.Tensor:
        out = self.bert(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            output_hidden_states=True,
        )
        hs = out.hidden_states
        K = self.text_last_k_layers
        layers_stack = torch.stack(hs[-K:], dim=0)
        w = torch.nn.functional.softmax(self.layer_mix_logits, dim=0).view(K, 1, 1, 1)
        h = (layers_stack * w).sum(dim=0)
        attn_mask = batch["attention_mask"]
        key_padding_mask = attn_mask == 0
        h_n = self.text_attn_norm(h)
        attn_out, _ = self.text_attn(
            h_n, h_n, h_n, key_padding_mask=key_padding_mask, need_weights=False
        )
        h = h + self.text_attn_dropout(attn_out)
        if self.text_pooling == "cls":
            z_txt = h[:, 0, :]
        else:
            mask = attn_mask.unsqueeze(-1).float()
            z_txt = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        z_cb = batch["pred_cb"]
        if z_cb.dim() == 1:
            z_cb = z_cb.unsqueeze(-1)
        emb_parts = [self.embeds[k](batch[f"emb_{k}"]) for k in self.embeds.keys()]
        z_emb = torch.cat(emb_parts, dim=1)
        z = torch.cat([z_txt, z_cb, z_emb], dim=1)
        return self.head(z).squeeze(-1)
def multimodal_bert_tail_param_groups(
    model: MultimodalSalaryModel,
    *,
    body_lr: float,
    tail_lr: float,
    head_lr: float,
    bert_wd: float,
    head_wd: float,
    tail_layer_count: int,
) -> list[dict]:
    """BERT: embeddings + нижние слои (body) vs последние K encoder-слоёв (tail); остальное — head."""
    layers = list(model.bert.encoder.layer)
    n = len(layers)
    k = max(1, min(int(tail_layer_count), n))
    tail_param_ids: set[int] = set()
    for lay in layers[n - k :]:
        for p in lay.parameters():
            tail_param_ids.add(id(p))
    body_bert: list[nn.Parameter] = []
    tail_bert: list[nn.Parameter] = []
    for p in model.bert.parameters():
        if id(p) in tail_param_ids:
            tail_bert.append(p)
        else:
            body_bert.append(p)
    bert_ids = {id(p) for p in model.bert.parameters()}
    head_params = [
        p for p in model.parameters() if id(p) not in bert_ids and p.requires_grad
    ]
    return [
        {"params": body_bert, "lr": body_lr, "weight_decay": bert_wd},
        {"params": tail_bert, "lr": tail_lr, "weight_decay": bert_wd},
        {"params": head_params, "lr": head_lr, "weight_decay": head_wd},
    ]
def fit_multimodal_train_loader(
    model: MultimodalSalaryModel,
    train_loader,
    loss_fn: nn.Module,
    *,
    n_epochs: int,
    head_lr: float,
    head_wd: float,
    body_lr: float,
    tail_lr: float,
    bert_wd: float,
    tail_layer_count: int,
    warmup_ratio: float,
    accum_steps: int,
    max_grad_norm: float,
    device: torch.device,
    tqdm_desc: str | None = None,
    tqdm_leave: bool = True,
) -> None:
    """Tail param groups + AMP (CUDA) + grad clip + gradient accumulation."""
    from tqdm.auto import tqdm
    use_amp = device.type == "cuda"
    optimizer = torch.optim.AdamW(
        multimodal_bert_tail_param_groups(
            model,
            body_lr=body_lr,
            tail_lr=tail_lr,
            head_lr=head_lr,
            bert_wd=bert_wd,
            head_wd=head_wd,
            tail_layer_count=tail_layer_count,
        )
    )
    accum_steps = max(1, int(accum_steps))
    steps_per_epoch = (len(train_loader) + accum_steps - 1) // accum_steps
    total_opt_steps = max(1, steps_per_epoch * n_epochs)
    warmup_steps = int(total_opt_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_opt_steps,
    )
    scaler = GradScaler(enabled=use_amp)
    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        micro = 0
        loader_it = train_loader
        if tqdm_desc is not None:
            loader_it = tqdm(
                train_loader,
                desc=f"{tqdm_desc} ep{epoch + 1}/{n_epochs}",
                leave=tqdm_leave,
            )
        for batch in loader_it:
            for k in batch:
                batch[k] = batch[k].to(device)
            with autocast(enabled=use_amp):
                pred = model(batch)
                loss = loss_fn(pred, batch["y"]) / accum_steps
            if use_amp:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            micro += 1
            if micro % accum_steps == 0:
                if use_amp:
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                if use_amp:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
        if micro % accum_steps != 0:
            if use_amp:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            if use_amp:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

In [ ]:

#Smoke: ruBERT + M2.10 CatBoost (pred_cb) + emb + MLP; tail LR + AMP + grad clip + accum + early stopping 
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.cuda.amp import GradScaler, autocast
from tqdm.auto import tqdm
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pathlib import Path
DEBUG_MAX_TRAIN = 4096  # None = весь train-фолда
N_EPOCHS_MAX = 12
EARLY_PATIENCE = 3
EARLY_MIN_DELTA = 1e-4
BATCH_SIZE = 32
ACCUM_STEPS = 2
BERT_TAIL_LAYERS = 4
BERT_BODY_LR = 2e-5
BERT_TAIL_LR = 5e-5
HEAD_LR = 1e-4
BERT_WD = 0.0
HEAD_WD = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
USE_AMP = DEVICE.type == "cuda"
TEXT_POOLING = "mean"  
TEXT_LAST_K_LAYERS = 4
TEXT_ATTN_HEADS = 8
CKPT_DIR = Path("checkpoints_m4_smoke")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
split_iter = list(gkf.split(X_pool, y_pool, groups))
tr, va = split_iter[0]
df_tr = df_train_pool.iloc[tr].reset_index(drop=True)
df_va = df_train_pool.iloc[va].reset_index(drop=True)
if DEBUG_MAX_TRAIN is not None:
    df_tr = df_tr.iloc[:DEBUG_MAX_TRAIN].reset_index(drop=True)
CB_PARAMS_SMOKE = default_m210_catboost_params()
CB_PARAMS_SMOKE["iterations"] = 400 
pred_cb_tr, pred_cb_va = catboost_m210_pred_for_train_val(
    df_tr,
    df_va,
    inner_splits=4,
    catboost_params=CB_PARAMS_SMOKE,
    oof_on_train=True,
)
emb_tr = {c: EMB_VOCABS_POOL[c].transform(df_tr[c]) for c in COLS_CAT_EMBED}
emb_va = {c: EMB_VOCABS_POOL[c].transform(df_va[c]) for c in COLS_CAT_EMBED}
tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
train_ds = VacancyFoldDataset(
    pred_cb_tr,
    df_tr["text_for_model"].tolist(),
    emb_tr,
    df_tr[TARGET].to_numpy(),
    tokenizer,
    MAX_LENGTH,
)
val_ds = VacancyFoldDataset(
    pred_cb_va,
    df_va["text_for_model"].tolist(),
    emb_va,
    df_va[TARGET].to_numpy(),
    tokenizer,
    MAX_LENGTH,
)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0
)
train_eval_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)
vocab_sizes = {c: len(EMB_VOCABS_POOL[c].token_to_id) for c in COLS_CAT_EMBED}
model = MultimodalSalaryModel(
    BERT_NAME,
    vocab_sizes=vocab_sizes,
    freeze_bert=False,
    text_pooling=TEXT_POOLING,
    text_last_k_layers=TEXT_LAST_K_LAYERS,
    text_attn_heads=TEXT_ATTN_HEADS,
).to(DEVICE)
optimizer = torch.optim.AdamW(
    multimodal_bert_tail_param_groups(
        model,
        body_lr=BERT_BODY_LR,
        tail_lr=BERT_TAIL_LR,
        head_lr=HEAD_LR,
        bert_wd=BERT_WD,
        head_wd=HEAD_WD,
        tail_layer_count=BERT_TAIL_LAYERS,
    )
)
loss_fn = torch.nn.MSELoss()
steps_per_epoch = (len(train_loader) + ACCUM_STEPS - 1) // ACCUM_STEPS
total_optimizer_steps = steps_per_epoch * N_EPOCHS_MAX
warmup_steps = int(total_optimizer_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_optimizer_steps,
)
scaler = GradScaler(enabled=USE_AMP)
epoch_pbar = tqdm(range(N_EPOCHS_MAX), desc="epoch", position=0)
history: list[dict] = []
best_val_rmse = float("inf")
patience = 0
best_state_cpu = None
best_epoch = 0
for epoch in epoch_pbar:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    micro = 0
    batch_pbar = tqdm(
        train_loader,
        desc=f"train {epoch + 1}/{N_EPOCHS_MAX}",
        leave=False,
        position=1,
    )
    for batch in batch_pbar:
        for k in batch:
            batch[k] = batch[k].to(DEVICE)
        with autocast(enabled=USE_AMP):
            pred = model(batch)
            loss = loss_fn(pred, batch["y"]) / ACCUM_STEPS
        if USE_AMP:
            scaler.scale(loss).backward()
        else:
            loss.backward()
        micro += 1
        if micro % ACCUM_STEPS == 0:
            if USE_AMP:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            if USE_AMP:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        batch_pbar.set_postfix(loss=f"{(loss.item() * ACCUM_STEPS):.4f}")
    if micro % ACCUM_STEPS != 0:
        if USE_AMP:
            scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        if USE_AMP:
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
    model.eval()
    tr_y, tr_p = [], []
    with torch.no_grad():
        for batch in tqdm(
            train_eval_loader,
            desc=f"train-eval {epoch + 1}/{N_EPOCHS_MAX}",
            leave=False,
            position=1,
        ):
            for k in batch:
                batch[k] = batch[k].to(DEVICE)
            pred = model(batch)
            tr_y.append(batch["y"].detach().cpu().numpy().reshape(-1))
            tr_p.append(pred.detach().cpu().numpy().reshape(-1))
    yt_tr = np.concatenate(tr_y)
    yp_tr = np.concatenate(tr_p)
    train_rmse = float(np.sqrt(mean_squared_error(yt_tr, yp_tr)))
    train_mae = float(mean_absolute_error(yt_tr, yp_tr))
    train_r2 = float(r2_score(yt_tr, yp_tr))
    val_y_true_parts: list[np.ndarray] = []
    val_y_pred_parts: list[np.ndarray] = []
    with torch.no_grad():
        val_pbar = tqdm(
            val_loader,
            desc=f"val {epoch + 1}/{N_EPOCHS_MAX}",
            leave=False,
            position=2,
        )
        for batch in val_pbar:
            for k in batch:
                batch[k] = batch[k].to(DEVICE)
            pred = model(batch)
            val_y_true_parts.append(batch["y"].detach().cpu().numpy().reshape(-1))
            val_y_pred_parts.append(pred.detach().cpu().numpy().reshape(-1))
    yt_va = np.concatenate(val_y_true_parts)
    yp_va = np.concatenate(val_y_pred_parts)
    val_rmse = float(np.sqrt(mean_squared_error(yt_va, yp_va)))
    val_mae = float(mean_absolute_error(yt_va, yp_va))
    val_r2 = float(r2_score(yt_va, yp_va))
    history.append(
        {
            "epoch": epoch + 1,
            "train_rmse": train_rmse,
            "train_mae": train_mae,
            "train_r2": train_r2,
            "val_rmse": val_rmse,
            "val_mae": val_mae,
            "val_r2": val_r2,
            "text_pooling": TEXT_POOLING,
        }
    )
    epoch_pbar.set_postfix(
        tr_rmse=f"{train_rmse:.4f}",
        va_rmse=f"{val_rmse:.4f}",
        va_mae=f"{val_mae:.4f}",
        va_r2=f"{val_r2:.4f}",
    )
    if val_rmse < best_val_rmse - EARLY_MIN_DELTA:
        best_val_rmse = val_rmse
        patience = 0
        best_epoch = epoch + 1
        best_state_cpu = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience += 1
        if patience >= EARLY_PATIENCE:
            print(
                f"Early stopping: val_rmse не улучшался {EARLY_PATIENCE} эпох (best={best_val_rmse:.4f})."
            )
            break
ckpt_path = CKPT_DIR / f"smoke_m210cb_{TEXT_POOLING}_k{TEXT_LAST_K_LAYERS}.pt"
if best_state_cpu is not None:
    torch.save(
        {
            "model_state_dict": best_state_cpu,
            "best_val_rmse": best_val_rmse,
            "best_epoch": best_epoch,
            "text_pooling": TEXT_POOLING,
            "text_last_k_layers": TEXT_LAST_K_LAYERS,
            "text_attn_heads": TEXT_ATTN_HEADS,
            "cb_iterations_smoke": CB_PARAMS_SMOKE.get("iterations"),
        },
        ckpt_path,
    )
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state_cpu.items()})
    print(f"Чекпоинт (лучший val_rmse={best_val_rmse:.6f}, эпоха={best_epoch}): {ckpt_path.resolve()}")
else:
    print("Чекпоинт не сохранён (нет улучшения val).")
print("Smoke (M2.10 CatBoost pred_cb + BERT + emb + MLP + val-checkpoint) завершён.")
display(pd.DataFrame(history))


In [ ]:
SEARCH_MAX_TRAIN = 4096
SEARCH_FOLD_INDICES = [0, 1, 2] 
HP_ACCUM_STEPS = 2
HP_MAX_GRAD_NORM = 1.0
BERT_TAIL_LAYERS_HP = 4
BERT_BODY_LR_HP = 2e-5
BERT_TAIL_LR_HP = 5e-5
BERT_WD_HP = 0.0
WARMUP_RATIO_HP = 0.1
# CatBoost M2.10: ускорение сетки HP (полные 1000 в default_m210_catboost_params)
HP_CB_PARAMS = default_m210_catboost_params()
HP_CB_PARAMS["iterations"] = 350
HP_CB_INNER_SPLITS = 3
HP_CANDIDATES = [
    {"lr": 3e-5,  "batch_size": 32, "n_epochs": 3, "weight_decay": 0.0},
    {"lr": 1e-4,  "batch_size": 32, "n_epochs": 3, "weight_decay": 0.01},
    {"lr": 3e-4,  "batch_size": 32, "n_epochs": 3, "weight_decay": 0.01},
    {"lr": 3e-4,  "batch_size": 16, "n_epochs": 3, "weight_decay": 0.01},
]
split_list = list(gkf.split(X_pool, y_pool, groups))
tokenizer_hp = AutoTokenizer.from_pretrained(BERT_NAME)
def train_eval_one_fold(fold_idx: int, hp: dict) -> dict:
    tr, va = split_list[fold_idx]
    df_tr = df_train_pool.iloc[tr].reset_index(drop=True)
    df_va = df_train_pool.iloc[va].reset_index(drop=True)
    if SEARCH_MAX_TRAIN is not None:
        df_tr = df_tr.iloc[:SEARCH_MAX_TRAIN].reset_index(drop=True)
    pred_cb_tr, pred_cb_va = catboost_m210_pred_for_train_val(
        df_tr,
        df_va,
        inner_splits=HP_CB_INNER_SPLITS,
        catboost_params=HP_CB_PARAMS,
        oof_on_train=True,
    )
    emb_tr = {c: EMB_VOCABS_POOL[c].transform(df_tr[c]) for c in COLS_CAT_EMBED}
    emb_va = {c: EMB_VOCABS_POOL[c].transform(df_va[c]) for c in COLS_CAT_EMBED}
    train_ds = VacancyFoldDataset(
        pred_cb_tr,
        df_tr["text_for_model"].tolist(),
        emb_tr,
        df_tr[TARGET].to_numpy(),
        tokenizer_hp,
        MAX_LENGTH,
    )
    val_ds = VacancyFoldDataset(
        pred_cb_va,
        df_va["text_for_model"].tolist(),
        emb_va,
        df_va[TARGET].to_numpy(),
        tokenizer_hp,
        MAX_LENGTH,
    )
    bs = int(hp["batch_size"])
    train_loader = DataLoader(
        train_ds, batch_size=bs, shuffle=True, collate_fn=collate_fn, num_workers=0
    )
    val_loader = DataLoader(
        val_ds, batch_size=bs, shuffle=False, collate_fn=collate_fn, num_workers=0
    )
    vocab_sizes = {c: len(EMB_VOCABS_POOL[c].token_to_id) for c in COLS_CAT_EMBED}
    model = MultimodalSalaryModel(
        BERT_NAME,
        vocab_sizes=vocab_sizes,
        freeze_bert=False,
        text_pooling="mean",
        text_last_k_layers=4,
        text_attn_heads=8,
    ).to(DEVICE)
    head_lr = float(hp["lr"])
    head_wd = float(hp["weight_decay"])
    n_ep = int(hp["n_epochs"])
    loss_fn = nn.MSELoss()
    fit_multimodal_train_loader(
        model,
        train_loader,
        loss_fn,
        n_epochs=n_ep,
        head_lr=head_lr,
        head_wd=head_wd,
        body_lr=BERT_BODY_LR_HP,
        tail_lr=BERT_TAIL_LR_HP,
        bert_wd=BERT_WD_HP,
        tail_layer_count=BERT_TAIL_LAYERS_HP,
        warmup_ratio=WARMUP_RATIO_HP,
        accum_steps=HP_ACCUM_STEPS,
        max_grad_norm=HP_MAX_GRAD_NORM,
        device=DEVICE,
        tqdm_desc=None,
    )
    model.eval()
    parts_y, parts_p = [], []
    with torch.no_grad():
        for batch in val_loader:
            for k in batch:
                batch[k] = batch[k].to(DEVICE)
            pred = model(batch)
            parts_y.append(batch["y"].detach().cpu().numpy().reshape(-1))
            parts_p.append(pred.detach().cpu().numpy().reshape(-1))
    yt = np.concatenate(parts_y)
    yp = np.concatenate(parts_p)
    out = {
        "fold": fold_idx,
        "val_rmse": float(np.sqrt(mean_squared_error(yt, yp))),
        "val_mae": float(mean_absolute_error(yt, yp)),
        "val_r2": float(r2_score(yt, yp)),
        "n_train": len(df_tr),
        "n_val": len(df_va),
        "m4_branch": "bert_catboost_emb_mlp",
    }
    del model, train_loader, val_loader, train_ds, val_ds
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return out
rows = []
for hp in tqdm(HP_CANDIDATES, desc="HP grid"):
    per_fold = [train_eval_one_fold(fi, hp) for fi in SEARCH_FOLD_INDICES]
    row = {
        "head_lr": hp["lr"],
        "batch_size": hp["batch_size"],
        "n_epochs": hp["n_epochs"],
        "head_weight_decay": hp["weight_decay"],
        "bert_body_lr": BERT_BODY_LR_HP,
        "bert_tail_lr": BERT_TAIL_LR_HP,
        "accum_steps": HP_ACCUM_STEPS,
        "val_rmse_mean": float(np.mean([x["val_rmse"] for x in per_fold])),
        "val_mae_mean": float(np.mean([x["val_mae"] for x in per_fold])),
        "val_r2_mean": float(np.mean([x["val_r2"] for x in per_fold])),
    }
    rows.append(row)
df_hp = pd.DataFrame(rows).sort_values("val_rmse_mean", ascending=True).reset_index(drop=True)
print("Лучший кандидат (минимальный val_rmse_mean):")
display(df_hp.head(5))
print("\nПолная таблица:")
display(df_hp)

In [ ]:
# Обучение: GroupKFold тот же тренер, что и в HP (tail BERT + AMP + clip + accum)
CV_N_EPOCHS = 3
CV_BATCH_SIZE = 16
CV_HEAD_LR = 3e-4
CV_HEAD_WD = 0.01
CV_BODY_LR = 2e-5
CV_TAIL_LR = 5e-5
CV_BERT_WD = 0.0
CV_TAIL_LAYERS = 4
CV_ACCUM_STEPS = 2
CV_MAX_GRAD_NORM = 1.0
CV_WARMUP_RATIO = 0.1
CV_MAX_TRAIN = 4096  # None = весь train-фолд
# CatBoost M2.10 на каждом внешнем фолде
CV_CB_PARAMS = default_m210_catboost_params()
CV_CB_PARAMS["iterations"] = 400
CV_CB_INNER_SPLITS = 3
tokenizer_cv = AutoTokenizer.from_pretrained(BERT_NAME)
fold_rows: list[dict] = []
for fold, (tr, va) in enumerate(gkf.split(X_pool, y_pool, groups)):
    df_tr = df_train_pool.iloc[tr].reset_index(drop=True)
    df_va = df_train_pool.iloc[va].reset_index(drop=True)
    if CV_MAX_TRAIN is not None:
        df_tr = df_tr.iloc[:CV_MAX_TRAIN].reset_index(drop=True)
    pred_cb_tr, pred_cb_va = catboost_m210_pred_for_train_val(
        df_tr,
        df_va,
        inner_splits=CV_CB_INNER_SPLITS,
        catboost_params=CV_CB_PARAMS,
        oof_on_train=True,
    )
    emb_tr = {c: EMB_VOCABS_POOL[c].transform(df_tr[c]) for c in COLS_CAT_EMBED}
    emb_va = {c: EMB_VOCABS_POOL[c].transform(df_va[c]) for c in COLS_CAT_EMBED}
    train_ds = VacancyFoldDataset(
        pred_cb_tr,
        df_tr["text_for_model"].tolist(),
        emb_tr,
        df_tr[TARGET].to_numpy(),
        tokenizer_cv,
        MAX_LENGTH,
    )
    val_ds = VacancyFoldDataset(
        pred_cb_va,
        df_va["text_for_model"].tolist(),
        emb_va,
        df_va[TARGET].to_numpy(),
        tokenizer_cv,
        MAX_LENGTH,
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=CV_BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=CV_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
    )
    vocab_sizes = {c: len(EMB_VOCABS_POOL[c].token_to_id) for c in COLS_CAT_EMBED}
    model = MultimodalSalaryModel(
        BERT_NAME,
        vocab_sizes=vocab_sizes,
        freeze_bert=False,
        text_pooling="mean",
        text_last_k_layers=4,
        text_attn_heads=8,
    ).to(DEVICE)
    loss_fn = nn.MSELoss()
    fit_multimodal_train_loader(
        model,
        train_loader,
        loss_fn,
        n_epochs=CV_N_EPOCHS,
        head_lr=CV_HEAD_LR,
        head_wd=CV_HEAD_WD,
        body_lr=CV_BODY_LR,
        tail_lr=CV_TAIL_LR,
        bert_wd=CV_BERT_WD,
        tail_layer_count=CV_TAIL_LAYERS,
        warmup_ratio=CV_WARMUP_RATIO,
        accum_steps=CV_ACCUM_STEPS,
        max_grad_norm=CV_MAX_GRAD_NORM,
        device=DEVICE,
        tqdm_desc=f"fold{fold}",
    )
    model.eval()
    val_y_true_parts: list[np.ndarray] = []
    val_y_pred_parts: list[np.ndarray] = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"fold {fold} val", leave=False):
            for k in batch:
                batch[k] = batch[k].to(DEVICE)
            pred = model(batch)
            val_y_true_parts.append(batch["y"].detach().cpu().numpy().reshape(-1))
            val_y_pred_parts.append(pred.detach().cpu().numpy().reshape(-1))
    yt = np.concatenate(val_y_true_parts)
    yp = np.concatenate(val_y_pred_parts)
    fold_rows.append(
        {
            "fold": fold,
            "val_rmse": float(np.sqrt(mean_squared_error(yt, yp))),
            "val_mae": float(mean_absolute_error(yt, yp)),
            "val_r2": float(r2_score(yt, yp)),
            "n_train": len(df_tr),
            "n_val": len(df_va),
            "m4_branch": "bert_catboost_emb_mlp",
            "cv_body_lr": CV_BODY_LR,
            "cv_tail_lr": CV_TAIL_LR,
            "cv_head_lr": CV_HEAD_LR,
            "cv_accum_steps": CV_ACCUM_STEPS,
        }
    )
    del model, train_loader, val_loader, train_ds, val_ds
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
df_cv = pd.DataFrame(fold_rows).set_index("fold")
display(df_cv)
summary = pd.DataFrame(
    {
        "mean": df_cv[["val_rmse", "val_mae", "val_r2"]].mean(),
        "std": df_cv[["val_rmse", "val_mae", "val_r2"]].std(ddof=0),
    }
)
print("\nCV val: mean ± std (последняя эпоха каждого фолда) ")
display(summary)

Виден большой разброс по фолдам.
При GroupKFold по региону такое часто означает географическую неоднородность: 
на одном наборе регионов в валидации распределение зарплат легче угадать (фолд 3), 
на другом сложнее или с другим сдвигом (фолд 0).

In [ ]:
FINAL_EPOCHS = 3
FINAL_BATCH = 16
FINAL_HEAD_LR = 3e-4
FINAL_HEAD_WD = 0.01
FINAL_BODY_LR = 2e-5
FINAL_TAIL_LR = 5e-5
FINAL_BERT_WD = 0.0
FINAL_TAIL_LAYERS = 4
FINAL_ACCUM_STEPS = 2
FINAL_MAX_GRAD_NORM = 1.0
FINAL_WARMUP_RATIO = 0.1
FINAL_TEXT_POOLING = "mean"
FINAL_TEXT_LAST_K_LAYERS = 4
FINAL_TEXT_ATTN_HEADS = 8
# CatBoost M2.10 на всём train-pool: OOF для pred_cb при обучении NN; для теста отдельный fit на pool
FINAL_CB_PARAMS = default_m210_catboost_params()
FINAL_CB_PARAMS["iterations"] = 1000 
FINAL_CB_INNER_SPLITS = 3
emb_all = {c: EMB_VOCABS_POOL[c].transform(df_train_pool[c]) for c in COLS_CAT_EMBED}
tokenizer_final = AutoTokenizer.from_pretrained(BERT_NAME)
# OOF pred_cb на train-pool (та же идея, что в catboost_m210_pred_for_train_val с oof_on_train=True)
pred_cb_all, _ = catboost_m210_pred_for_train_val(
    df_train_pool,
    df_train_pool.iloc[:1],
    inner_splits=FINAL_CB_INNER_SPLITS,
    catboost_params=FINAL_CB_PARAMS,
    oof_on_train=True,
)
train_final_ds = VacancyFoldDataset(
    pred_cb_all,
    df_train_pool["text_for_model"].tolist(),
    emb_all,
    df_train_pool[TARGET].to_numpy(),
    tokenizer_final,
    MAX_LENGTH,
)
train_final_loader = DataLoader(
    train_final_ds,
    batch_size=FINAL_BATCH,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
)
vocab_sizes = {c: len(EMB_VOCABS_POOL[c].token_to_id) for c in COLS_CAT_EMBED}
model_final = MultimodalSalaryModel(
    BERT_NAME,
    vocab_sizes=vocab_sizes,
    freeze_bert=False,
    text_pooling=FINAL_TEXT_POOLING,
    text_last_k_layers=FINAL_TEXT_LAST_K_LAYERS,
    text_attn_heads=FINAL_TEXT_ATTN_HEADS,
).to(DEVICE)
loss_fn = nn.MSELoss()
fit_multimodal_train_loader(
    model_final,
    train_final_loader,
    loss_fn,
    n_epochs=FINAL_EPOCHS,
    head_lr=FINAL_HEAD_LR,
    head_wd=FINAL_HEAD_WD,
    body_lr=FINAL_BODY_LR,
    tail_lr=FINAL_TAIL_LR,
    bert_wd=FINAL_BERT_WD,
    tail_layer_count=FINAL_TAIL_LAYERS,
    warmup_ratio=FINAL_WARMUP_RATIO,
    accum_steps=FINAL_ACCUM_STEPS,
    max_grad_norm=FINAL_MAX_GRAD_NORM,
    device=DEVICE,
    tqdm_desc="final",
)
# --- Оценка на отложенном тесте ---
df_te = df_test
emb_te = {c: EMB_VOCABS_POOL[c].transform(df_te[c]) for c in COLS_CAT_EMBED}
X_pool_fit, cat_idx = _cb_pool_X(df_train_pool)
cb_test = CatBoostRegressor(**FINAL_CB_PARAMS)
cb_test.fit(
    Pool(X_pool_fit, label=df_train_pool[TARGET].to_numpy(), cat_features=cat_idx)
)
X_te, _ = _cb_pool_X(df_te)
pred_cb_te = cb_test.predict(X_te).astype(np.float32)
test_ds = VacancyFoldDataset(
    pred_cb_te,
    df_te["text_for_model"].tolist(),
    emb_te,
    df_te[TARGET].to_numpy(),
    tokenizer_final,
    MAX_LENGTH,
)
test_loader = DataLoader(
    test_ds, batch_size=FINAL_BATCH, shuffle=False, collate_fn=collate_fn, num_workers=0
)
model_final.eval()
ty, tp = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="test"):
        for k in batch:
            batch[k] = batch[k].to(DEVICE)
        pred = model_final(batch)
        ty.append(batch["y"].detach().cpu().numpy().reshape(-1))
        tp.append(pred.detach().cpu().numpy().reshape(-1))
yt = np.concatenate(ty)
yp = np.concatenate(tp)
print("TEST RMSE:", float(np.sqrt(mean_squared_error(yt, yp))))
print("TEST MAE:", float(mean_absolute_error(yt, yp)))
print("TEST R2:", float(r2_score(yt, yp)))

In [ ]:
import json
from pathlib import Path
import torch
ARTIFACT_DIR = Path("artifacts_m4_final")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
# 1) CatBoost (модель для pred_cb на новых табличных признаках)
cb_path = ARTIFACT_DIR / "catboost_m210_test.cbm"
cb_test.save_model(str(cb_path))
print("Saved CatBoost:", cb_path.resolve())
#  2) Словари эмбеддингов (token -> id; при загрузке соберите EmbeddingVocab(col, token_to_id)) 
emb_vocab_payload = {c: EMB_VOCABS_POOL[c].token_to_id for c in COLS_CAT_EMBED}
emb_vocab_path = ARTIFACT_DIR / "embedding_vocabs.json"
with open(emb_vocab_path, "w", encoding="utf-8") as f:
    json.dump(emb_vocab_payload, f, ensure_ascii=False, indent=2)
print("Saved embedding vocabs:", emb_vocab_path.resolve())
#  3) Метаданные для восстановления MultimodalSalaryModel и воспроизводимости 
def _json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)
meta = {
    "bert_name": BERT_NAME,
    "max_length": MAX_LENGTH,
    "cols_cat_embed": list(COLS_CAT_EMBED),
    "vocab_sizes": {c: int(len(EMB_VOCABS_POOL[c].token_to_id)) for c in COLS_CAT_EMBED},
    "model_kwargs": {
        "text_pooling": FINAL_TEXT_POOLING,
        "text_last_k_layers": int(FINAL_TEXT_LAST_K_LAYERS),
        "text_attn_heads": int(FINAL_TEXT_ATTN_HEADS),
    },
    "train_hparams": {
        "final_epochs": int(FINAL_EPOCHS),
        "final_batch": int(FINAL_BATCH),
        "final_head_lr": float(FINAL_HEAD_LR),
        "final_head_wd": float(FINAL_HEAD_WD),
        "final_body_lr": float(FINAL_BODY_LR),
        "final_tail_lr": float(FINAL_TAIL_LR),
        "final_bert_wd": float(FINAL_BERT_WD),
        "final_tail_layers": int(FINAL_TAIL_LAYERS),
        "final_accum_steps": int(FINAL_ACCUM_STEPS),
        "final_max_grad_norm": float(FINAL_MAX_GRAD_NORM),
        "final_warmup_ratio": float(FINAL_WARMUP_RATIO),
    },
    "catboost_train_params": _json_safe(dict(FINAL_CB_PARAMS)),
    "catboost_inner_splits_final_oof": int(FINAL_CB_INNER_SPLITS),
    "target_col": TARGET,
    "m4_branch": "bert_catboost_emb_mlp",
}
config_path = ARTIFACT_DIR / "m4_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Saved config:", config_path.resolve())
# --- 4) Веса NN (state_dict на CPU; целиком модуль не сохраняем — переносимее между средами) ---
nn_ckpt_path = ARTIFACT_DIR / "m4_multimodal_state.pt"
torch.save(
    {
        "model_state_dict": {k: v.detach().cpu() for k, v in model_final.state_dict().items()},
        "meta": meta,
    },
    nn_ckpt_path,
)
print("Saved NN state_dict:", nn_ckpt_path.resolve())

In [52]:
#интерпретация ошибки в рублях
from sklearn.metrics import mean_absolute_error, mean_squared_error
S_true = np.expm1(yt.astype(np.float64))
S_pred = np.expm1(yp.astype(np.float64))
MAE_rub = mean_absolute_error(S_true, S_pred)
RMSE_rub = float(np.sqrt(mean_squared_error(S_true, S_pred)))
print("MAE (руб., salary_from_adj):", MAE_rub)
print("RMSE (руб., salary_from_adj):", RMSE_rub)

MAE (руб., salary_from_adj): 13007.808174310316
RMSE (руб., salary_from_adj): 19725.00868004375


In [53]:
median_ae_rub = float(np.median(np.abs(S_pred - S_true)))
print("Median |error| (руб.):", median_ae_rub)

Median |error| (руб.): 8094.53050511871


In [54]:
#Проанализируем ошибки по квантилям зарплаты 
assert len(yt) == len(df_te), "Длина yt не совпадает с df_te — проверьте порядок строк и предсказаний."
S_true = np.expm1(np.asarray(yt, dtype=np.float64))
S_pred = np.expm1(np.asarray(yp, dtype=np.float64))
abs_err = np.abs(S_pred - S_true)
sgn_err = S_pred - S_true  # > 0: модель завысила salary_from_adj в рублях
def rmse_rub(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    return float(np.sqrt(np.mean((a - b) ** 2)))
q = 5  # квинтили; для декилей поставьте q=10
df_q = pd.DataFrame({"S_true": S_true, "S_pred": S_pred, "abs_err": abs_err, "sgn_err": sgn_err})
df_q["salary_quantile_bin"] = pd.qcut(df_q["S_true"], q=q, duplicates="drop")
rows = []
for bin_interval, sub in df_q.groupby("salary_quantile_bin", observed=True):
    rows.append(
        {
            "bin_S_true_rub": str(bin_interval),
            "n": int(len(sub)),
            "S_true_min_rub": float(sub["S_true"].min()),
            "S_true_max_rub": float(sub["S_true"].max()),
            "S_true_median_rub": float(sub["S_true"].median()),
            "MAE_rub": float(sub["abs_err"].mean()),
            "RMSE_rub": rmse_rub(sub["S_true"].to_numpy(), sub["S_pred"].to_numpy()),
            "bias_mean_rub": float(sub["sgn_err"].mean()),
        }
    )
summary_quantiles = pd.DataFrame(rows)
display(summary_quantiles)
print("Всего тест:", len(df_q))
print("MAE_rub (весь тест):", float(abs_err.mean()))
print("RMSE_rub (весь тест):", rmse_rub(S_true, S_pred))

,bin_S_true_rub,n,S_true_min_rub,S_true_max_rub,S_true_median_rub,MAE_rub,RMSE_rub,bias_mean_rub
0,"(19241.996, 39999.997]",5665,19241.997289,39999.997102,34100.010309,8215.040811,12478.434694,7444.752591
1,"(39999.997, 50000.009]",5338,40009.993107,50000.009267,45976.988269,8769.068869,13824.474253,7391.678721
2,"(50000.009, 60999.985]",4824,50016.987894,60999.984756,57199.998279,10835.445675,15861.178583,7858.441801
3,"(60999.985, 84999.982]",5233,61100.011450,84999.982210,70000.031651,14004.602628,19291.083309,7170.129832
4,"(84999.982, 459769.909]",5254,85056.988785,459769.908629,114943.042142,23483.763417,31383.574908,-1869.316823


Всего тест: 26314
MAE_rub (весь тест): 13007.808174310316
RMSE_rub (весь тест): 19725.00868004375


In [ ]:
assert len(yt) == len(df_te)
S_true = np.expm1(np.asarray(yt, dtype=np.float64))
S_pred = np.expm1(np.asarray(yp, dtype=np.float64))
resid_rub = S_pred - S_true
abs_err = np.abs(resid_rub)
df_r = pd.DataFrame(
    {
        "region_name": df_te["region_name"].astype(str).values,
        "abs_err_rub": abs_err,
        "resid_rub": resid_rub,
        "S_true_rub": S_true,
    }
)
by_region = (
    df_r.groupby("region_name", dropna=False)
    .agg(
        n=("abs_err_rub", "count"),
        MAE_rub=("abs_err_rub", "mean"),
        median_abs_err_rub=("abs_err_rub", "median"),
        bias_mean_rub=("resid_rub", "mean"),
        bias_median_rub=("resid_rub", "median"),
        S_true_median_rub=("S_true_rub", "median"),
    )
    .sort_values("MAE_rub", ascending=False)
)
print(" Топ-30 регионов по MAE_rub")
display(by_region.head(30))
vc = df_r["region_name"].value_counts()
print("Регионов всего:", int(vc.shape[0]))
print("Доля наблюдений из регионов с n < 50:", float((df_r["region_name"].map(vc) < 50).mean()))